# CAMELS-US 2026 revision — Step 1 v5.7
## TRAIN FIRST — selected HPO epoch per Add8 model — 3 concurrent runs on GPU 0

This version fixes the execution logic.

### Stage A — training (runs now)
For each of the 8 locked additional models:

- 10 original random seeds;
- train only to the model's original **HPO-selected epoch**;
- **no validation during training**;
- save model/optimizer only at the selected final epoch;
- use the persisted NeuralHydrology `train_data.p`;
- preserve the uploaded Top-10 final configuration for every field that is not model-specific, seed/run-specific, path-specific, or part of this requested final-training protocol;
- use NeuralHydrology's native `schedule_runs`;
- 3 simultaneous runs on GPU 0.

### Stage B — after all training finishes
Validation is run **once** from the saved selected-epoch checkpoint, followed by test from the same checkpoint.

Raw CAMELS evaluation paths are needed only for Stage B and therefore **must not block Stage A training**.

### Exact selected epochs
- rank 10 `1206_203556` → 20
- rank 11 `0511_234753` → 25
- rank 13 `0609_031111` → 20
- rank 14 `2109_134554` → 25
- rank 15 `1410_002534` → 30
- rank 16 `2903_024612` → 25
- rank 17 `1503_082400` → 30
- rank 18 `1708_155828` → 30

Total: **80 runs = 20 runs to epoch 20 + 30 runs to epoch 25 + 30 runs to epoch 30.**


In [2]:
from __future__ import annotations

import os
import sys
import re
import json
import yaml
import time
import shutil
import subprocess
from copy import deepcopy
from pathlib import Path
from typing import Any, Dict, Mapping

import numpy as np
import pandas as pd

NH_SOURCE = Path(r"C:\Users\hosseinifh\neuralhydrology")
if not NH_SOURCE.exists():
    raise FileNotFoundError(f"NeuralHydrology checkout not found: {NH_SOURCE}")

if str(NH_SOURCE) not in sys.path:
    sys.path.insert(0, str(NH_SOURCE))

# Child processes launched by NeuralHydrology's scheduler must use the same
# conda Python environment as this notebook kernel.
kernel_python_dir = str(Path(sys.executable).resolve().parent)
path_parts = [p for p in os.environ.get("PATH", "").split(os.pathsep) if p]
path_parts = [p for p in path_parts if p != kernel_python_dir]
os.environ["PATH"] = os.pathsep.join([kernel_python_dir, *path_parts])

pp = [p for p in os.environ.get("PYTHONPATH", "").split(os.pathsep) if p]
pp = [p for p in pp if p != str(NH_SOURCE)]
os.environ["PYTHONPATH"] = os.pathsep.join([str(NH_SOURCE), *pp])

import torch
import neuralhydrology
from neuralhydrology.utils.config import Config
from neuralhydrology.nh_run_scheduler import schedule_runs

print("Python            :", sys.executable)
print("NeuralHydrology   :", Path(neuralhydrology.__file__).resolve())
print("PyTorch           :", torch.__version__)
print("CUDA available    :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU 0             :", torch.cuda.get_device_name(0))


Python            : c:\Users\hosseinifh\AppData\Local\anaconda3\envs\MyPy_env2025\python.exe
NeuralHydrology   : C:\Users\hosseinifh\neuralhydrology\neuralhydrology\__init__.py
PyTorch           : 2.7.1+cu118
CUDA available    : True
GPU 0             : NVIDIA GeForce RTX 3060


## 1. Paths and execution switches

In [3]:
PROJECT_HOME = Path(r"F:\Experiments\CAMELS_US")
ROOT = PROJECT_HOME / "Clean_4_upload"

# Existing reusable NeuralHydrology cached training dataset.
TRAIN_DATA_DIR = ROOT / "Train_Data"
TRAIN_DATA_FILE = TRAIN_DATA_DIR / "train_data" / "train_data.p"
STATIC_ATTRIBUTES_DIR = TRAIN_DATA_DIR / "camels_attributes_v2.0"

BASIN_FILE = ROOT / "531_basin_list.txt"

REVISION_ROOT = ROOT / "Revision_2026_Top18_4Framework"

# New unique directories: do not reuse any v5.5/v5.6 'final30' output.
CONFIG_DIR = REVISION_ROOT / "configs" / "top18_additional8_selected_epoch_train_v57"
RUN_DIR = REVISION_ROOT / "runs" / "top18_additional8_selected_epoch_train_v57"
AUDIT_DIR = REVISION_ROOT / "audit"

for d in [CONFIG_DIR, RUN_DIR, AUDIT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

GPU_IDS = [0]
RUNS_PER_GPU = 3

SEEDS = [191889, 307443, 328237, 385445, 473461,
         643969, 665980, 845563, 985957, 993061]

GENERATE_CONFIGS = True

# Keep False until all preflight cells PASS.
RUN_TRAINING = False

# Stage B remains OFF during training setup.
RUN_POSTTRAIN_EVALUATION = False
EVAL_DATA_DIR_OVERRIDE = None

assert TRAIN_DATA_FILE.is_file(), TRAIN_DATA_FILE
assert STATIC_ATTRIBUTES_DIR.is_dir(), STATIC_ATTRIBUTES_DIR
assert BASIN_FILE.is_file(), BASIN_FILE

with BASIN_FILE.open("r", encoding="utf-8") as f:
    BASINS = [x.strip() for x in f if x.strip()]

assert len(BASINS) == 531
assert len(set(BASINS)) == 531
assert all(re.fullmatch(r"\d{8}", b) for b in BASINS)

print("TRAIN_DATA_FILE      :", TRAIN_DATA_FILE)
print("STATIC_ATTRIBUTES_DIR:", STATIC_ATTRIBUTES_DIR)
print("BASIN_FILE           :", BASIN_FILE)
print("Basins               :", len(BASINS))
print("CONFIG_DIR           :", CONFIG_DIR)
print("RUN_DIR              :", RUN_DIR)
print("GPU_IDS              :", GPU_IDS)
print("RUNS_PER_GPU         :", RUNS_PER_GPU)
print("TRAINING PATH LOCK: PASS")


TRAIN_DATA_FILE      : F:\Experiments\CAMELS_US\Clean_4_upload\Train_Data\train_data\train_data.p
STATIC_ATTRIBUTES_DIR: F:\Experiments\CAMELS_US\Clean_4_upload\Train_Data\camels_attributes_v2.0
BASIN_FILE           : F:\Experiments\CAMELS_US\Clean_4_upload\531_basin_list.txt
Basins               : 531
CONFIG_DIR           : F:\Experiments\CAMELS_US\Clean_4_upload\Revision_2026_Top18_4Framework\configs\top18_additional8_selected_epoch_train_v57
RUN_DIR              : F:\Experiments\CAMELS_US\Clean_4_upload\Revision_2026_Top18_4Framework\runs\top18_additional8_selected_epoch_train_v57
GPU_IDS              : [0]
RUNS_PER_GPU         : 3
TRAINING PATH LOCK: PASS


## 2. Uploaded Top-10 config is the canonical template

The uploaded Top-10 final config is kept as the baseline.

During Stage A, the deliberate protocol changes are:

- `epochs = selected_epoch`
- `validate_every = None` → no validation inside training
- `save_weights_every = selected_epoch` → save only the final selected checkpoint

Validation is not removed from the experiment; it is moved to **Stage B after training**, exactly as requested.


In [3]:
ORIGINAL_TOP10_SAMPLE = {'batch_size': 128, 'cache_validation_data': True, 'clip_gradient_norm': 1, 'clip_targets_to_zero': ['QObs(mm/d)'], 'commit_hash': None, 'data_dir': '/home/projects/nhlstm/neural-files/DataBank/camels-data', 'dataset': 'camels_us', 'device': 'cuda:0', 'dynamic_inputs': ['prcp(mm/day)_daymet', 'srad(W/m2)_daymet', 'tmax(C)_daymet', 'tmin(C)_daymet', 'vp(Pa)_daymet', 'PRCP(mm/day)_nldas', 'SRAD(W/m2)_nldas', 'Tmax(C)_nldas', 'Tmin(C)_nldas', 'Vp(Pa)_nldas', 'PRCP(mm/day)_maurer', 'SRAD(W/m2)_maurer', 'Tmax(C)_maurer', 'Tmin(C)_maurer', 'Vp(Pa)_maurer'], 'epochs': 30, 'experiment_name': '531_basins_RandomSearch', 'forcings': ['daymet', 'maurer', 'nldas'], 'head': 'regression', 'hidden_size': 256, 'initial_forget_bias': 3, 'learning_rate': {0: 0.001, 10: 0.0005, 25: 0.001}, 'log_interval': 5, 'log_tensorboard': True, 'loss': 'RMSE', 'metrics': ['NSE', 'KGE', 'MSE', 'RMSE', 'Alpha-NSE', 'Beta-NSE', 'Beta-KGE', 'Pearson-r', 'FHV', 'FMS', 'FLV', 'Peak-Timing', 'Peak-MAPE', 'Missed-Peaks'], 'model': 'cudalstm', 'num_workers': 1, 'number_of_basins': 531, 'optimizer': 'Adam', 'output_activation': 'linear', 'output_dropout': 0.4, 'predict_last_n': 1, 'save_validation_results': True, 'save_weights_every': 1, 'seed': 473461, 'seq_length': 450, 'static_attributes': ['elev_mean', 'slope_mean', 'area_gages2', 'frac_forest', 'lai_max', 'lai_diff', 'gvf_max', 'gvf_diff', 'soil_depth_pelletier', 'soil_depth_statsgo', 'soil_porosity', 'soil_conductivity', 'max_water_content', 'sand_frac', 'silt_frac', 'clay_frac', 'carbonate_rocks_frac', 'geol_permeability', 'p_mean', 'pet_mean', 'aridity', 'frac_snow', 'high_prec_freq', 'high_prec_dur', 'low_prec_freq', 'low_prec_dur'], 'target_noise_std': 0.05, 'target_variables': ['QObs(mm/d)'], 'test_basin_file': '/home/projects/nhlstm/neural-files/DataBank/camels-data/531_basin_list.txt', 'test_end_date': '30/09/1999', 'test_start_date': '01/10/1989', 'train_basin_file': '/home/projects/nhlstm/neural-files/DataBank/camels-data/531_basin_list.txt', 'train_data_file': '/home/projects/nhlstm/neural-files/DataBank/camels-data/train_data/train_data.p', 'train_end_date': '30/09/2008', 'train_start_date': '01/10/1999', 'validate_every': 1, 'validate_n_random_basins': -1, 'validation_basin_file': '/home/projects/nhlstm/neural-files/DataBank/camels-data/531_basin_list.txt', 'validation_end_date': '30/09/1989', 'validation_start_date': '01/10/1980'}

TOP10_TEMPLATE = deepcopy(ORIGINAL_TOP10_SAMPLE)

print("Original Top-10 sample:")
for k in [
    "epochs", "validate_every", "save_weights_every",
    "validate_n_random_basins", "save_validation_results",
    "cache_validation_data", "num_workers", "model", "optimizer"
]:
    print(f"{k}: {TOP10_TEMPLATE.get(k)}")

assert TOP10_TEMPLATE["epochs"] == 30
assert TOP10_TEMPLATE["validate_every"] == 1
assert TOP10_TEMPLATE["save_weights_every"] == 1
assert TOP10_TEMPLATE["num_workers"] == 1
assert TOP10_TEMPLATE["model"] == "cudalstm"
assert TOP10_TEMPLATE["optimizer"] == "Adam"

print("TOP10 TEMPLATE READ: PASS")


Original Top-10 sample:
epochs: 30
validate_every: 1
save_weights_every: 1
validate_n_random_basins: -1
save_validation_results: True
cache_validation_data: True
num_workers: 1
model: cudalstm
optimizer: Adam
TOP10 TEMPLATE READ: PASS


## 3. Locked Add8 configurations and original selected epochs

In [4]:
LOCKED8 = [{'global_rank': 10, 'token': '1206_203556', 'selected_epoch': 20, 'seq_length': 365, 'batch_size': 128, 'hidden_size': 256, 'output_dropout': 0.2, 'initial_forget_bias': -3, 'target_noise_std': 0.0, 'lr0': 0.001, 'lr10': 0.0005, 'lr25': 0.0001, 'loss': 'RMSE', 'SFmean_NSE': 0.462343, 'SFmean_KGE': 0.664592, 'AvgMetrics': 0.563468}, {'global_rank': 11, 'token': '0511_234753', 'selected_epoch': 25, 'seq_length': 365, 'batch_size': 64, 'hidden_size': 256, 'output_dropout': 0.0, 'initial_forget_bias': 0, 'target_noise_std': 0.1, 'lr0': 0.001, 'lr10': 0.0005, 'lr25': 0.0001, 'loss': 'RMSE', 'SFmean_NSE': 0.439393, 'SFmean_KGE': 0.68614, 'AvgMetrics': 0.562767}, {'global_rank': 13, 'token': '0609_031111', 'selected_epoch': 20, 'seq_length': 365, 'batch_size': 32, 'hidden_size': 128, 'output_dropout': 0.2, 'initial_forget_bias': 1, 'target_noise_std': 0.0, 'lr0': 0.001, 'lr10': 0.0005, 'lr25': 0.001, 'loss': 'RMSE', 'SFmean_NSE': 0.435557, 'SFmean_KGE': 0.649234, 'AvgMetrics': 0.542395}, {'global_rank': 14, 'token': '2109_134554', 'selected_epoch': 25, 'seq_length': 120, 'batch_size': 32, 'hidden_size': 256, 'output_dropout': 0.2, 'initial_forget_bias': -3, 'target_noise_std': 0.02, 'lr0': 0.001, 'lr10': 0.0005, 'lr25': 0.0001, 'loss': 'RMSE', 'SFmean_NSE': 0.441362, 'SFmean_KGE': 0.643304, 'AvgMetrics': 0.542333}, {'global_rank': 15, 'token': '1410_002534', 'selected_epoch': 30, 'seq_length': 365, 'batch_size': 128, 'hidden_size': 256, 'output_dropout': 0.2, 'initial_forget_bias': 1, 'target_noise_std': 0.05, 'lr0': 0.001, 'lr10': 0.0005, 'lr25': 0.001, 'loss': 'RMSE', 'SFmean_NSE': 0.415874, 'SFmean_KGE': 0.658187, 'AvgMetrics': 0.53703}, {'global_rank': 16, 'token': '2903_024612', 'selected_epoch': 25, 'seq_length': 365, 'batch_size': 32, 'hidden_size': 128, 'output_dropout': 0.0, 'initial_forget_bias': 5, 'target_noise_std': 0.02, 'lr0': 0.001, 'lr10': 0.0005, 'lr25': 0.0001, 'loss': 'RMSE', 'SFmean_NSE': 0.41975, 'SFmean_KGE': 0.651371, 'AvgMetrics': 0.535561}, {'global_rank': 17, 'token': '1503_082400', 'selected_epoch': 30, 'seq_length': 450, 'batch_size': 128, 'hidden_size': 128, 'output_dropout': 0.2, 'initial_forget_bias': 3, 'target_noise_std': 0.01, 'lr0': 0.001, 'lr10': 0.0005, 'lr25': 0.001, 'loss': 'RMSE', 'SFmean_NSE': 0.457711, 'SFmean_KGE': 0.60867, 'AvgMetrics': 0.533191}, {'global_rank': 18, 'token': '1708_155828', 'selected_epoch': 30, 'seq_length': 1460, 'batch_size': 512, 'hidden_size': 256, 'output_dropout': 0.2, 'initial_forget_bias': -3, 'target_noise_std': 0.0, 'lr0': 0.001, 'lr10': 0.001, 'lr25': 0.0001, 'loss': 'RMSE', 'SFmean_NSE': 0.416533, 'SFmean_KGE': 0.648084, 'AvgMetrics': 0.532309}]

locked8_df = pd.DataFrame(LOCKED8).sort_values("global_rank").reset_index(drop=True)

assert len(locked8_df) == 8
assert locked8_df["global_rank"].tolist() == [10, 11, 13, 14, 15, 16, 17, 18]
assert locked8_df["selected_epoch"].tolist() == [20, 25, 20, 25, 30, 25, 30, 30]
assert set(locked8_df["loss"]) == {"RMSE"}

display(
    locked8_df[
        [
            "global_rank", "token", "selected_epoch",
            "SFmean_NSE", "SFmean_KGE", "AvgMetrics",
            "seq_length", "batch_size", "hidden_size",
            "output_dropout", "initial_forget_bias",
            "target_noise_std", "lr0", "lr10", "lr25", "loss"
        ]
    ]
)

print("ADD8 + SELECTED EPOCH LOCK: PASS")
print("Model counts by selected epoch:")
print(locked8_df["selected_epoch"].value_counts().sort_index())


,global_rank,token,selected_epoch,SFmean_NSE,SFmean_KGE,AvgMetrics,seq_length,batch_size,hidden_size,output_dropout,initial_forget_bias,target_noise_std,lr0,lr10,lr25,loss
0,10,1206_203556,20,0.462343,0.664592,0.563468,365,128,256,0.2,-3,0.00,0.001,0.0005,0.0001,RMSE
1,11,0511_234753,25,0.439393,0.686140,0.562767,365,64,256,0.0,0,0.10,0.001,0.0005,0.0001,RMSE
2,13,0609_031111,20,0.435557,0.649234,0.542395,365,32,128,0.2,1,0.00,0.001,0.0005,0.0010,RMSE
3,14,2109_134554,25,0.441362,0.643304,0.542333,120,32,256,0.2,-3,0.02,0.001,0.0005,0.0001,RMSE
4,15,1410_002534,30,0.415874,0.658187,0.537030,365,128,256,0.2,1,0.05,0.001,0.0005,0.0010,RMSE
5,16,2903_024612,25,0.419750,0.651371,0.535561,365,32,128,0.0,5,0.02,0.001,0.0005,0.0001,RMSE
6,17,1503_082400,30,0.457711,0.608670,0.533191,450,128,128,0.2,3,0.01,0.001,0.0005,0.0010,RMSE
7,18,1708_155828,30,0.416533,0.648084,0.532309,1460,512,256,0.2,-3,0.00,0.001,0.0010,0.0001,RMSE


ADD8 + SELECTED EPOCH LOCK: PASS
Model counts by selected epoch:
selected_epoch
20    2
25    3
30    3
Name: count, dtype: int64


## 4. Generate 80 training configs

The generated launch config starts from the Top-10 template.

Allowed differences are only:

- exact Add8 HPO fields;
- model-specific selected epoch;
- seed and experiment name;
- local paths/device/run directory;
- `validate_every=None`;
- `save_weights_every=selected_epoch`.


In [5]:
HPO_FIELDS = {
    "seq_length", "batch_size", "hidden_size",
    "output_dropout", "initial_forget_bias",
    "target_noise_std", "learning_rate", "loss",
}

RUN_FIELDS = {
    "seed", "experiment_name", "device",
    "data_dir", "train_basin_file",
    "validation_basin_file", "test_basin_file",
    "train_data_file", "run_dir",
}

PROTOCOL_FIELDS = {
    "epochs", "validate_every", "save_weights_every",
}

ALLOWED_DIFFERENCES = HPO_FIELDS | RUN_FIELDS | PROTOCOL_FIELDS


def write_yaml(cfg: Mapping[str, Any], path: Path):
    with path.open("w", encoding="utf-8") as f:
        yaml.safe_dump(dict(cfg), f, sort_keys=False, allow_unicode=True)


def build_training_config(hp: Mapping[str, Any], seed: int) -> Dict[str, Any]:
    cfg = deepcopy(TOP10_TEMPLATE)

    # ------------------------------------
    # Exact model-specific HPO settings
    # ------------------------------------
    cfg["seq_length"] = int(hp["seq_length"])
    cfg["batch_size"] = int(hp["batch_size"])
    cfg["hidden_size"] = int(hp["hidden_size"])
    cfg["output_dropout"] = float(hp["output_dropout"])
    cfg["initial_forget_bias"] = int(hp["initial_forget_bias"])
    cfg["target_noise_std"] = float(hp["target_noise_std"])
    cfg["learning_rate"] = {
        0: float(hp["lr0"]),
        10: float(hp["lr10"]),
        25: float(hp["lr25"]),
    }
    cfg["loss"] = str(hp["loss"])

    # ------------------------------------
    # Selected-epoch final training
    # ------------------------------------
    selected_epoch = int(hp["selected_epoch"])
    cfg["epochs"] = selected_epoch

    # IMPORTANT: validation is a separate post-training evaluation stage.
    cfg["validate_every"] = None

    # Save only one model/optimizer checkpoint: selected final epoch.
    cfg["save_weights_every"] = selected_epoch

    # ------------------------------------
    # Run identity
    # ------------------------------------
    cfg["seed"] = int(seed)
    cfg["experiment_name"] = (
        f"Top18_Add8_rank{int(hp['global_rank']):02d}_"
        f"{hp['token']}_SE{selected_epoch:02d}_RS{int(seed)}"
    )

    # ------------------------------------
    # Training paths
    #
    # train_data.p supplies the cached time series.
    # data_dir only needs to provide static attributes during Stage A.
    # ------------------------------------
    cfg["device"] = "cuda:0"
    cfg["data_dir"] = str(TRAIN_DATA_DIR)
    cfg["train_data_file"] = str(TRAIN_DATA_FILE)

    cfg["train_basin_file"] = str(BASIN_FILE)
    cfg["validation_basin_file"] = str(BASIN_FILE)
    cfg["test_basin_file"] = str(BASIN_FILE)

    cfg["run_dir"] = str(RUN_DIR)

    return cfg


processed_dir = CONFIG_DIR / "processed"

if GENERATE_CONFIGS:
    if processed_dir.exists() and list(processed_dir.glob("*.yml")):
        raise RuntimeError(
            "Processed configs already exist in this v5.7 directory. "
            "Training has started before; set GENERATE_CONFIGS=False to resume."
        )

    # Fresh v5.7 pending configs.
    for p in CONFIG_DIR.glob("*.yml"):
        p.unlink()

    records = []

    for hp in LOCKED8:
        for seed in SEEDS:
            cfg = build_training_config(hp, seed)

            fn = (
                f"Top18_Add8_rank{int(hp['global_rank']):02d}_"
                f"{hp['token']}_SE{int(hp['selected_epoch']):02d}_RS{int(seed)}.yml"
            )
            path = CONFIG_DIR / fn
            write_yaml(cfg, path)

            records.append({
                "global_rank": int(hp["global_rank"]),
                "token": str(hp["token"]),
                "selected_epoch": int(hp["selected_epoch"]),
                "seed": int(seed),
                "config_file": str(path),
            })

    manifest = (
        pd.DataFrame(records)
        .sort_values(["global_rank", "seed"])
        .reset_index(drop=True)
    )

    manifest.to_csv(
        AUDIT_DIR / "v57_add8_selected_epoch_80run_manifest.csv",
        index=False,
    )
else:
    manifest = pd.read_csv(
        AUDIT_DIR / "v57_add8_selected_epoch_80run_manifest.csv"
    )

assert len(manifest) == 80
assert manifest["token"].nunique() == 8
assert manifest["seed"].nunique() == 10
assert set(manifest["seed"].astype(int)) == set(SEEDS)

run_epoch_counts = (
    manifest["selected_epoch"].astype(int)
    .value_counts()
    .sort_index()
    .to_dict()
)
assert run_epoch_counts == {20: 20, 25: 30, 30: 30}, run_epoch_counts

per_model = (
    manifest.groupby(["global_rank", "token", "selected_epoch"])["seed"]
    .agg(n_runs="size", n_unique_seeds="nunique")
    .reset_index()
)
assert len(per_model) == 8
assert (per_model["n_runs"] == 10).all()
assert (per_model["n_unique_seeds"] == 10).all()

print("80 TRAINING CONFIGS: PASS")
display(per_model)
print("Run distribution:", run_epoch_counts)


80 TRAINING CONFIGS: PASS


,global_rank,token,selected_epoch,n_runs,n_unique_seeds
0,10,1206_203556,20,10,10
1,11,0511_234753,25,10,10
2,13,0609_031111,20,10,10
3,14,2109_134554,25,10,10
4,15,1410_002534,30,10,10
5,16,2903_024612,25,10,10
6,17,1503_082400,30,10,10
7,18,1708_155828,30,10,10


Run distribution: {20: 20, 25: 30, 30: 30}


## 5. Strict preflight — no raw evaluation data required

In [6]:
audit_rows = []

expected_epoch_by_token = {
    str(x["token"]): int(x["selected_epoch"])
    for x in LOCKED8
}

for _, row in manifest.iterrows():
    path = Path(row["config_file"])
    cfg = yaml.safe_load(path.read_text(encoding="utf-8"))

    token = str(row["token"])
    selected_epoch = expected_epoch_by_token[token]

    # Every Top-10 field outside the allowed differences must be identical.
    unexpected = {}
    for key, old_value in ORIGINAL_TOP10_SAMPLE.items():
        if key in ALLOWED_DIFFERENCES:
            continue
        if cfg.get(key) != old_value:
            unexpected[key] = {
                "top10": old_value,
                "new": cfg.get(key),
            }

    nh_ok = True
    nh_error = None
    try:
        _ = Config(path)
    except Exception as exc:
        nh_ok = False
        nh_error = repr(exc)

    audit_rows.append({
        "global_rank": int(row["global_rank"]),
        "token": token,
        "seed": int(row["seed"]),
        "selected_epoch": selected_epoch,
        "top10_constants_identical": len(unexpected) == 0,
        "unexpected": json.dumps(unexpected, default=str),
        "epochs_correct": int(cfg["epochs"]) == selected_epoch,
        "validation_disabled_during_training": cfg.get("validate_every") is None,
        "only_final_checkpoint_requested": int(cfg["save_weights_every"]) == selected_epoch,
        "cached_train_file_correct":
            Path(str(cfg["train_data_file"])).resolve() == TRAIN_DATA_FILE.resolve(),
        "training_data_dir_correct":
            Path(str(cfg["data_dir"])).resolve() == TRAIN_DATA_DIR.resolve(),
        "num_workers_same_as_top10":
            int(cfg["num_workers"]) == int(ORIGINAL_TOP10_SAMPLE["num_workers"]),
        "nh_config_parse_ok": nh_ok,
        "nh_error": nh_error,
    })

audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv(
    AUDIT_DIR / "v57_strict_preflight_80.csv",
    index=False
)

checks = [
    "top10_constants_identical",
    "epochs_correct",
    "validation_disabled_during_training",
    "only_final_checkpoint_requested",
    "cached_train_file_correct",
    "training_data_dir_correct",
    "num_workers_same_as_top10",
    "nh_config_parse_ok",
]

for col in checks:
    if not audit_df[col].all():
        display(audit_df.loc[~audit_df[col]])
        raise RuntimeError(f"PRE-FLIGHT FAILED: {col}")

assert TRAIN_DATA_FILE.is_file()
assert STATIC_ATTRIBUTES_DIR.is_dir()
assert BASIN_FILE.is_file()

print("STRICT TRAINING PRE-FLIGHT: PASS")
print("80/80 preserve Top-10 constants outside allowed fields.")
print("80/80 use exact selected HPO epoch.")
print("80/80 have validate_every=None during training.")
print("80/80 save weights only at selected epoch.")
print("80/80 use cached train_data.p.")
print("80/80 use Train_Data for static attributes.")
print("80/80 parse with NeuralHydrology Config.")
print("RAW CAMELS FORCING/STREAMFLOW IS NOT REQUIRED FOR STAGE A.")


STRICT TRAINING PRE-FLIGHT: PASS
80/80 preserve Top-10 constants outside allowed fields.
80/80 use exact selected HPO epoch.
80/80 have validate_every=None during training.
80/80 save weights only at selected epoch.
80/80 use cached train_data.p.
80/80 use Train_Data for static attributes.
80/80 parse with NeuralHydrology Config.
RAW CAMELS FORCING/STREAMFLOW IS NOT REQUIRED FOR STAGE A.


## 6. Inspect representative configs for epoch 20, 25 and 30

In [7]:
for epoch in [20, 25, 30]:
    row = manifest.loc[manifest["selected_epoch"].astype(int) == epoch].iloc[0]
    path = Path(row["config_file"])
    cfg = yaml.safe_load(path.read_text(encoding="utf-8"))

    print("\n" + "=" * 100)
    print("Representative selected epoch:", epoch)
    print("File:", path.name)
    print("token:", row["token"])
    print("seed:", row["seed"])
    print("epochs:", cfg["epochs"])
    print("validate_every:", cfg["validate_every"])
    print("save_weights_every:", cfg["save_weights_every"])
    print("train_data_file:", cfg["train_data_file"])
    print("data_dir:", cfg["data_dir"])
    print("learning_rate:", cfg["learning_rate"])
    print("loss:", cfg["loss"])



Representative selected epoch: 20
File: Top18_Add8_rank10_1206_203556_SE20_RS191889.yml
token: 1206_203556
seed: 191889
epochs: 20
validate_every: None
save_weights_every: 20
train_data_file: F:\Experiments\CAMELS_US\Clean_4_upload\Train_Data\train_data\train_data.p
data_dir: F:\Experiments\CAMELS_US\Clean_4_upload\Train_Data
learning_rate: {0: 0.001, 10: 0.0005, 25: 0.0001}
loss: RMSE

Representative selected epoch: 25
File: Top18_Add8_rank11_0511_234753_SE25_RS191889.yml
token: 0511_234753
seed: 191889
epochs: 25
validate_every: None
save_weights_every: 25
train_data_file: F:\Experiments\CAMELS_US\Clean_4_upload\Train_Data\train_data\train_data.p
data_dir: F:\Experiments\CAMELS_US\Clean_4_upload\Train_Data
learning_rate: {0: 0.001, 10: 0.0005, 25: 0.0001}
loss: RMSE

Representative selected epoch: 30
File: Top18_Add8_rank15_1410_002534_SE30_RS191889.yml
token: 1410_002534
seed: 191889
epochs: 30
validate_every: None
save_weights_every: 30
train_data_file: F:\Experiments\CAMELS_US\Cl

## 7. Final GPU/scheduler check

In [5]:
assert torch.cuda.is_available()
assert GPU_IDS == [0]
assert RUNS_PER_GPU == 3

pending = sorted(CONFIG_DIR.glob("*.yml"))
processed = (
    sorted((CONFIG_DIR / "processed").glob("*.yml"))
    if (CONFIG_DIR / "processed").exists()
    else []
)

assert len(pending) + len(processed) == 35, (
    len(pending), len(processed)
)

print("FINAL PRE-TRAIN CHECK: PASS")
print("Pending configs       :", len(pending))
print("Processed configs     :", len(processed))
print("Total                 :", len(pending) + len(processed))
print("Concurrent train runs :", 3)
print("GPU                   :", 0)
print("Epoch 20 runs         :", 20)
print("Epoch 25 runs         :", 30)
print("Epoch 30 runs         :", 30)
print("Validation in Train   : NONE")
print("Checkpoint per run    : selected epoch ONLY")
print("\nSet RUN_TRAINING=True only after this PASS.")


AssertionError: (36, 35)

## 8. Launch Stage A training

This is the only cell that launches training.

NeuralHydrology's own scheduler is used exactly as requested:

```python
schedule_runs(
    mode="train",
    directory=CONFIG_DIR,
    gpu_ids=[0],
    runs_per_gpu=3,
)
```

After the scheduler returns, the notebook verifies that all 80 run directories exist and that every run contains exactly one model checkpoint at its own selected epoch.


In [6]:
# ============================================================
# FINAL PRE-TRAIN CHECK — STAGE A1
# 36 active non-rank18 + 35 processed + 9 held rank18 = 80
# ============================================================

pending = sorted(CONFIG_DIR.glob("*.yml"))

processed_dir = CONFIG_DIR / "processed"
processed = (
    sorted(processed_dir.glob("*.yml"))
    if processed_dir.exists()
    else []
)

hold_dir = CONFIG_DIR / "HOLD_rank18"
held = (
    sorted(hold_dir.glob("*.yml"))
    if hold_dir.exists()
    else []
)

print("=" * 100)
print("FINAL PRE-TRAIN CHECK — STAGE A1")
print("=" * 100)

print("Pending active configs :", len(pending))
print("Processed configs      :", len(processed))
print("Held rank18 configs    :", len(held))
print("Total accounted for    :", len(pending) + len(processed) + len(held))

# ------------------------------------------------------------
# Exact audited counts
# ------------------------------------------------------------

assert len(pending) == 36, (
    f"Expected 36 active configs, found {len(pending)}"
)

assert len(processed) == 35, (
    f"Expected 35 processed configs, found {len(processed)}"
)

assert len(held) == 9, (
    f"Expected 9 held rank18 configs, found {len(held)}"
)

assert len(pending) + len(processed) + len(held) == 80, (
    len(pending), len(processed), len(held)
)

# ------------------------------------------------------------
# Safety: NO rank18 may remain in active queue
# ------------------------------------------------------------

active_rank18 = [
    p.name for p in pending
    if "rank18_" in p.name
]

assert len(active_rank18) == 0, (
    "Rank18 config(s) still present in active queue:\n"
    + "\n".join(active_rank18)
)

# ------------------------------------------------------------
# Safety: held configs MUST all be rank18
# ------------------------------------------------------------

non_rank18_in_hold = [
    p.name for p in held
    if "rank18_" not in p.name
]

assert len(non_rank18_in_hold) == 0, (
    "Non-rank18 config(s) found in HOLD_rank18:\n"
    + "\n".join(non_rank18_in_hold)
)

print("\nFINAL PRE-TRAIN CHECK: PASS")
print("36 non-rank18 configs ready for Stage A1.")
print("35 runs already processed.")
print("9 rank18 configs safely held.")
print("80/80 configurations accounted for.")

FINAL PRE-TRAIN CHECK — STAGE A1
Pending active configs : 36
Processed configs      : 35
Held rank18 configs    : 9
Total accounted for    : 80

FINAL PRE-TRAIN CHECK: PASS
36 non-rank18 configs ready for Stage A1.
35 runs already processed.
9 rank18 configs safely held.
80/80 configurations accounted for.


In [7]:
RUN_TRAINING = True
print("RUN_TRAINING =", RUN_TRAINING)

RUN_TRAINING = True


In [ ]:
def find_run_dirs(run_root: Path) -> list[Path]:
    if not run_root.exists():
        return []
    return sorted([
        p for p in run_root.iterdir()
        if p.is_dir() and (p / "config.yml").is_file()
    ])


def selected_epoch_from_run_config(run_dir: Path) -> int:
    cfg = Config(run_dir / "config.yml")
    return int(cfg.epochs)


if RUN_TRAINING:
    print("=" * 100)
    print("STARTING STAGE A TRAINING")
    print("80 runs; 3 simultaneous runs on GPU 0")
    print("=" * 100)

    schedule_runs(
        mode="train",
        directory=CONFIG_DIR,
        gpu_ids=[0],
        runs_per_gpu=3,
    )

    run_dirs = find_run_dirs(RUN_DIR)
    print("\nNative NH scheduler returned.")
    print("Run directories found:", len(run_dirs))

    if len(run_dirs) != 80:
        raise RuntimeError(
            f"Expected 80 completed run directories; found {len(run_dirs)}. "
            "Do not start evaluation. Inspect run output.log files."
        )

    checkpoint_errors = []

    for rd in run_dirs:
        e = selected_epoch_from_run_config(rd)
        expected_model = f"model_epoch{e:03d}.pt"
        expected_optimizer = f"optimizer_state_epoch{e:03d}.pt"

        model_files = sorted(x.name for x in rd.glob("model_epoch*.pt"))
        optimizer_files = sorted(x.name for x in rd.glob("optimizer_state_epoch*.pt"))

        if model_files != [expected_model]:
            checkpoint_errors.append(
                (str(rd), "model", e, model_files)
            )

        if optimizer_files != [expected_optimizer]:
            checkpoint_errors.append(
                (str(rd), "optimizer", e, optimizer_files)
            )

    if checkpoint_errors:
        print("Checkpoint audit failures:")
        for x in checkpoint_errors[:40]:
            print(x)
        raise RuntimeError(
            f"{len(checkpoint_errors)} checkpoint audit problem(s) found."
        )

    print("POST-TRAIN CHECKPOINT AUDIT: PASS")
    print("80/80 runs trained to their own selected epoch.")
    print("80/80 contain exactly one model checkpoint.")
    print("80/80 contain exactly one optimizer checkpoint.")
    print("\nStage A is complete.")
    print("Next: Stage B one-time validation, then test.")
else:
    print("RUN_TRAINING=False — training has NOT started.")
    print("After FINAL PRE-TRAIN CHECK: PASS, set RUN_TRAINING=True and rerun this cell.")


STARTING STAGE A TRAINING
80 runs; 3 simultaneous runs on GPU 0
Starting run 1/80: python C:\Users\hosseinifh\neuralhydrology\neuralhydrology\nh_run.py train --config-file F:\Experiments\CAMELS_US\Clean_4_upload\Revision_2026_Top18_4Framework\configs\top18_additional8_selected_epoch_train_v57\Top18_Add8_rank11_0511_234753_SE25_RS328237.yml --gpu 0
Starting run 2/80: python C:\Users\hosseinifh\neuralhydrology\neuralhydrology\nh_run.py train --config-file F:\Experiments\CAMELS_US\Clean_4_upload\Revision_2026_Top18_4Framework\configs\top18_additional8_selected_epoch_train_v57\Top18_Add8_rank10_1206_203556_SE20_RS665980.yml --gpu 0
Starting run 3/80: python C:\Users\hosseinifh\neuralhydrology\neuralhydrology\nh_run.py train --config-file F:\Experiments\CAMELS_US\Clean_4_upload\Revision_2026_Top18_4Framework\configs\top18_additional8_selected_epoch_train_v57\Top18_Add8_rank10_1206_203556_SE20_RS845563.yml --gpu 0
Finished run python C:\Users\hosseinifh\neuralhydrology\neuralhydrology\nh_run

In [ ]:
def find_run_dirs(run_root: Path) -> list[Path]:
    if not run_root.exists():
        return []
    return sorted([
        p for p in run_root.iterdir()
        if p.is_dir() and (p / "config.yml").is_file()
    ])


def selected_epoch_from_run_config(run_dir: Path) -> int:
    cfg = Config(run_dir / "config.yml")
    return int(cfg.epochs)


if RUN_TRAINING:
    # ------------------------------------------------------------------
    # Stage A1: complete all remaining NON-rank18 runs
    # Current audited state:
    #   35 already completed
    #   36 active non-rank18 configs
    #    9 rank18 configs held separately
    # ------------------------------------------------------------------

    active_configs = sorted(CONFIG_DIR.glob("*.yml"))

    print("=" * 100)
    print("STARTING STAGE A1 TRAINING — NON-RANK18 ONLY")
    print(f"Active configs found: {len(active_configs)}")
    print("Expected active configs: 36")
    print("3 simultaneous runs on GPU 0")
    print("Rank18 configs remain on HOLD")
    print("=" * 100)

    if len(active_configs) != 36:
        raise RuntimeError(
            f"Expected exactly 36 active non-rank18 configs, "
            f"but found {len(active_configs)}. "
            "Do not start training until this is inspected."
        )

    # Safety check: no rank18 config may be present in active CONFIG_DIR
    rank18_active = [
        p for p in active_configs
        if "rank18_" in p.name
    ]

    if rank18_active:
        raise RuntimeError(
            "Rank18 config(s) found in active training directory:\n"
            + "\n".join(str(p) for p in rank18_active)
        )

    print("\nPRE-TRAIN AUDIT: PASS")
    print("36 active configs.")
    print("0 active rank18 configs.")
    print("\nStarting scheduler...\n")

    schedule_runs(
        mode="train",
        directory=CONFIG_DIR,
        gpu_ids=[0],
        runs_per_gpu=3,
    )

    print("\nNative NH scheduler returned.")

    run_dirs = find_run_dirs(RUN_DIR)

    print("Completed run directories found:", len(run_dirs))
    print("Expected after Stage A1: 71")

    if len(run_dirs) != 71:
        raise RuntimeError(
            f"Expected 71 completed run directories after Stage A1; "
            f"found {len(run_dirs)}. "
            "Do not start rank18 training or evaluation. "
            "Inspect run output.log files."
        )

    checkpoint_errors = []

    for rd in run_dirs:
        e = selected_epoch_from_run_config(rd)

        expected_model = f"model_epoch{e:03d}.pt"
        expected_optimizer = f"optimizer_state_epoch{e:03d}.pt"

        model_files = sorted(x.name for x in rd.glob("model_epoch*.pt"))
        optimizer_files = sorted(
            x.name for x in rd.glob("optimizer_state_epoch*.pt")
        )

        if model_files != [expected_model]:
            checkpoint_errors.append(
                (str(rd), "model", e, model_files)
            )

        if optimizer_files != [expected_optimizer]:
            checkpoint_errors.append(
                (str(rd), "optimizer", e, optimizer_files)
            )

    if checkpoint_errors:
        print("\nCheckpoint audit failures:")
        for x in checkpoint_errors[:40]:
            print(x)

        raise RuntimeError(
            f"{len(checkpoint_errors)} checkpoint audit problem(s) found."
        )

    print("\n" + "=" * 100)
    print("STAGE A1 COMPLETE")
    print("=" * 100)
    print("71/80 total runs completed.")
    print("71/71 completed runs passed checkpoint audit.")
    print("9 rank18 runs remain on HOLD.")
    print("\nDO NOT START EVALUATION YET.")
    print("Next: Stage A2 — rank18 training with reduced concurrency.")

else:
    print("RUN_TRAINING=False — training has NOT started.")

STARTING STAGE A1 TRAINING — NON-RANK18 ONLY
Active configs found: 36
Expected active configs: 36
3 simultaneous runs on GPU 0
Rank18 configs remain on HOLD

PRE-TRAIN AUDIT: PASS
36 active configs.
0 active rank18 configs.

Starting scheduler...

Starting run 1/36: python C:\Users\hosseinifh\neuralhydrology\neuralhydrology\nh_run.py train --config-file F:\Experiments\CAMELS_US\Clean_4_upload\Revision_2026_Top18_4Framework\configs\top18_additional8_selected_epoch_train_v57\Top18_Add8_rank15_1410_002534_SE30_RS385445.yml --gpu 0
Starting run 2/36: python C:\Users\hosseinifh\neuralhydrology\neuralhydrology\nh_run.py train --config-file F:\Experiments\CAMELS_US\Clean_4_upload\Revision_2026_Top18_4Framework\configs\top18_additional8_selected_epoch_train_v57\Top18_Add8_rank14_2109_134554_SE25_RS473461.yml --gpu 0
Starting run 3/36: python C:\Users\hosseinifh\neuralhydrology\neuralhydrology\nh_run.py train --config-file F:\Experiments\CAMELS_US\Clean_4_upload\Revision_2026_Top18_4Framework\c

In [2]:
# ============================================================
# CAMELS-US TOP18 — STAGE A1 SAFE RESUME
# Current audited state after crash:
#   47 processed
#   24 active non-rank18
#    9 rank18 held
# Run with 2 simultaneous jobs on GPU 0
# ============================================================

from pathlib import Path
import sys
import torch

from neuralhydrology.nh_run_scheduler import schedule_runs

CONFIG_DIR = Path(
    r"F:\Experiments\CAMELS_US\Clean_4_upload"
    r"\Revision_2026_Top18_4Framework"
    r"\configs\top18_additional8_selected_epoch_train_v57"
)

RUN_DIR = Path(
    r"F:\Experiments\CAMELS_US\Clean_4_upload"
    r"\Revision_2026_Top18_4Framework"
    r"\runs\top18_additional8_selected_epoch_train_v57"
)

PROCESSED_DIR = CONFIG_DIR / "processed"
HOLD_DIR = CONFIG_DIR / "HOLD_rank18"

# ------------------------------------------------------------
# 0. ENVIRONMENT SAFETY
# ------------------------------------------------------------

print("=" * 100)
print("ENVIRONMENT CHECK")
print("=" * 100)

print("Python         :", sys.executable)
print("PyTorch        :", torch.__version__)
print("CUDA available :", torch.cuda.is_available())

assert "WindowsApps" not in str(sys.executable), (
    f"Wrong Python environment: {sys.executable}"
)

assert torch.cuda.is_available(), "CUDA is not available."

print("GPU 0          :", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# 1. CURRENT FILESYSTEM ACCOUNTING
# ------------------------------------------------------------

pending = sorted(CONFIG_DIR.glob("*.yml"))
processed = sorted(PROCESSED_DIR.glob("*.yml"))
held = sorted(HOLD_DIR.glob("*.yml"))

print("\n" + "=" * 100)
print("CURRENT STAGE-A ACCOUNTING")
print("=" * 100)

print("Processed          :", len(processed))
print("Pending non-rank18 :", len(pending))
print("Held rank18        :", len(held))
print("Total              :", len(processed) + len(pending) + len(held))

assert len(processed) == 47, (
    f"Expected 47 processed, found {len(processed)}"
)

assert len(pending) == 24, (
    f"Expected 24 pending, found {len(pending)}"
)

assert len(held) == 9, (
    f"Expected 9 held rank18, found {len(held)}"
)

assert len(processed) + len(pending) + len(held) == 80

# ------------------------------------------------------------
# 2. RANK18 MUST REMAIN OUT OF ACTIVE QUEUE
# ------------------------------------------------------------

active_rank18 = [
    p.name for p in pending
    if "rank18_" in p.name
]

assert not active_rank18, (
    "Rank18 configs found in active queue:\n"
    + "\n".join(active_rank18)
)

bad_hold = [
    p.name for p in held
    if "rank18_" not in p.name
]

assert not bad_hold, (
    "Non-rank18 config found in HOLD_rank18:\n"
    + "\n".join(bad_hold)
)

print("\nPRE-TRAIN AUDIT: PASS")
print("47 completed runs preserved.")
print("24 non-rank18 runs ready.")
print("9 rank18 runs safely held.")
print("Concurrency: 2")
print("GPU: 0")

# ------------------------------------------------------------
# 3. NATIVE NEURALHYDROLOGY RESUME
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("STARTING STAGE A1 RESUME")
print("24 pending runs | 2 simultaneous | GPU 0")
print("=" * 100)

schedule_runs(
    mode="train",
    directory=CONFIG_DIR,
    gpu_ids=[0],
    runs_per_gpu=2,
)

# ------------------------------------------------------------
# 4. POST-SCHEDULER ACCOUNTING
# ------------------------------------------------------------

pending_after = sorted(CONFIG_DIR.glob("*.yml"))
processed_after = sorted(PROCESSED_DIR.glob("*.yml"))
held_after = sorted(HOLD_DIR.glob("*.yml"))

print("\n" + "=" * 100)
print("SCHEDULER RETURNED — FILESYSTEM STATUS")
print("=" * 100)

print("Processed          :", len(processed_after))
print("Pending non-rank18 :", len(pending_after))
print("Held rank18        :", len(held_after))
print(
    "Total              :",
    len(processed_after) + len(pending_after) + len(held_after)
)

assert len(processed_after) + len(pending_after) + len(held_after) == 80
assert len(held_after) == 9

if len(pending_after) == 0 and len(processed_after) == 71:
    print("\nSTAGE A1 COMPLETE: 71/80")
    print("Only the 9 held rank18 runs remain.")
else:
    print("\nStage A1 is not yet fully complete.")
    print("Any YAML still in CONFIG_DIR remains safely pending.")

ENVIRONMENT CHECK
Python         : c:\Users\hosseinifh\AppData\Local\anaconda3\envs\MyPy_env2025\python.exe
PyTorch        : 2.7.1+cu118
CUDA available : True
GPU 0          : NVIDIA GeForce RTX 3060

CURRENT STAGE-A ACCOUNTING
Processed          : 47
Pending non-rank18 : 24
Held rank18        : 9
Total              : 80

PRE-TRAIN AUDIT: PASS
47 completed runs preserved.
24 non-rank18 runs ready.
9 rank18 runs safely held.
Concurrency: 2
GPU: 0

STARTING STAGE A1 RESUME
24 pending runs | 2 simultaneous | GPU 0
Starting run 1/24: python C:\Users\hosseinifh\neuralhydrology\neuralhydrology\nh_run.py train --config-file F:\Experiments\CAMELS_US\Clean_4_upload\Revision_2026_Top18_4Framework\configs\top18_additional8_selected_epoch_train_v57\Top18_Add8_rank11_0511_234753_SE25_RS665980.yml --gpu 0
Starting run 2/24: python C:\Users\hosseinifh\neuralhydrology\neuralhydrology\nh_run.py train --config-file F:\Experiments\CAMELS_US\Clean_4_upload\Revision_2026_Top18_4Framework\configs\top18_addi

In [3]:
from pathlib import Path
import sys
import torch
import shutil

from neuralhydrology.nh_run_scheduler import schedule_runs

CONFIG_DIR = Path(
    r"F:\Experiments\CAMELS_US\Clean_4_upload"
    r"\Revision_2026_Top18_4Framework"
    r"\configs\top18_additional8_selected_epoch_train_v57"
)

PROCESSED_DIR = CONFIG_DIR / "processed"
HOLD_DIR = CONFIG_DIR / "HOLD_rank18"

print("=" * 100)
print("RANK18 RELEASE + TRAIN")
print("=" * 100)

print("Python         :", sys.executable)
print("PyTorch        :", torch.__version__)
print("CUDA available :", torch.cuda.is_available())
print("GPU 0          :", torch.cuda.get_device_name(0))

assert torch.cuda.is_available()
assert "WindowsApps" not in str(sys.executable)

# ------------------------------------------------------------
# 1. PRE-CHECK
# ------------------------------------------------------------

processed = sorted(PROCESSED_DIR.glob("*.yml"))
active = sorted(CONFIG_DIR.glob("*.yml"))
held = sorted(HOLD_DIR.glob("*.yml"))

print("\nBefore release:")
print("Processed :", len(processed))
print("Active    :", len(active))
print("Held r18  :", len(held))
print("Total     :", len(processed) + len(active) + len(held))

assert len(processed) == 71, f"Expected 71 processed, found {len(processed)}"
assert len(active) == 0, f"Expected 0 active configs, found {len(active)}"
assert len(held) == 9, f"Expected 9 held rank18 configs, found {len(held)}"

for p in held:
    assert "rank18_" in p.name, f"Unexpected config in HOLD: {p.name}"

# ------------------------------------------------------------
# 2. RELEASE RANK18 TO ACTIVE QUEUE
# ------------------------------------------------------------

print("\nReleasing rank18 configs...")

for src in held:
    dst = CONFIG_DIR / src.name

    assert not dst.exists(), f"Destination already exists: {dst}"

    shutil.move(str(src), str(dst))
    print("  released:", src.name)

# ------------------------------------------------------------
# 3. CHECK AFTER RELEASE
# ------------------------------------------------------------

processed = sorted(PROCESSED_DIR.glob("*.yml"))
active = sorted(CONFIG_DIR.glob("*.yml"))
held = sorted(HOLD_DIR.glob("*.yml"))

print("\nAfter release:")
print("Processed :", len(processed))
print("Active    :", len(active))
print("Held r18  :", len(held))
print("Total     :", len(processed) + len(active) + len(held))

assert len(processed) == 71
assert len(active) == 9
assert len(held) == 0
assert len(processed) + len(active) == 80

print("\n" + "=" * 100)
print("STARTING RANK18 TRAINING")
print("9 runs | 2 simultaneous | GPU 0")
print("=" * 100)

# ------------------------------------------------------------
# 4. TRAIN 2 AT A TIME
# ------------------------------------------------------------

schedule_runs(
    mode="train",
    directory=CONFIG_DIR,
    gpu_ids=[0],
    runs_per_gpu=2,
)

# ------------------------------------------------------------
# 5. FINAL CHECK
# ------------------------------------------------------------

processed_final = sorted(PROCESSED_DIR.glob("*.yml"))
active_final = sorted(CONFIG_DIR.glob("*.yml"))

print("\n" + "=" * 100)
print("FINAL STATUS")
print("=" * 100)

print("Processed :", len(processed_final))
print("Active    :", len(active_final))
print("Total     :", len(processed_final) + len(active_final))

if len(processed_final) == 80 and len(active_final) == 0:
    print("\nALL 80 TRAINING RUNS COMPLETE.")
else:
    print("\nSome rank18 runs are still pending.")

RANK18 RELEASE + TRAIN
Python         : c:\Users\hosseinifh\AppData\Local\anaconda3\envs\MyPy_env2025\python.exe
PyTorch        : 2.7.1+cu118
CUDA available : True
GPU 0          : NVIDIA GeForce RTX 3060

Before release:
Processed : 71
Active    : 0
Held r18  : 9
Total     : 80

Releasing rank18 configs...
  released: Top18_Add8_rank18_1708_155828_SE30_RS191889.yml
  released: Top18_Add8_rank18_1708_155828_SE30_RS307443.yml
  released: Top18_Add8_rank18_1708_155828_SE30_RS328237.yml
  released: Top18_Add8_rank18_1708_155828_SE30_RS385445.yml
  released: Top18_Add8_rank18_1708_155828_SE30_RS473461.yml
  released: Top18_Add8_rank18_1708_155828_SE30_RS643969.yml
  released: Top18_Add8_rank18_1708_155828_SE30_RS845563.yml
  released: Top18_Add8_rank18_1708_155828_SE30_RS985957.yml
  released: Top18_Add8_rank18_1708_155828_SE30_RS993061.yml

After release:
Processed : 71
Active    : 9
Held r18  : 0
Total     : 80

STARTING RANK18 TRAINING
9 runs | 2 simultaneous | GPU 0
Starting run 1/9: p

KeyboardInterrupt: 

In [1]:
from pathlib import Path
import subprocess
import os

CONFIG_DIR = Path(
    r"F:\Experiments\CAMELS_US\Clean_4_upload"
    r"\Revision_2026_Top18_4Framework"
    r"\configs\top18_additional8_selected_epoch_train_v57"
)

RUN_DIR = Path(
    r"F:\Experiments\CAMELS_US\Clean_4_upload"
    r"\Revision_2026_Top18_4Framework"
    r"\runs\top18_additional8_selected_epoch_train_v57"
)

PROCESSED_DIR = CONFIG_DIR / "processed"
HOLD_DIR = CONFIG_DIR / "HOLD_rank18"

print("=" * 100)
print("RANK18 POST-STOP AUDIT")
print("=" * 100)

# ------------------------------------------------------------------
# 1. Filesystem accounting
# ------------------------------------------------------------------

processed = sorted(PROCESSED_DIR.glob("*.yml"))
active = sorted(CONFIG_DIR.glob("*.yml"))
held = sorted(HOLD_DIR.glob("*.yml"))

processed_r18 = [p for p in processed if "rank18_" in p.name]
active_r18 = [p for p in active if "rank18_" in p.name]
held_r18 = [p for p in held if "rank18_" in p.name]

processed_non18 = [p for p in processed if "rank18_" not in p.name]
active_non18 = [p for p in active if "rank18_" not in p.name]
held_non18 = [p for p in held if "rank18_" not in p.name]

print("\nFILESYSTEM:")
print(f"Processed total       : {len(processed)}")
print(f"  non-rank18          : {len(processed_non18)}")
print(f"  rank18              : {len(processed_r18)}")
print(f"Active total          : {len(active)}")
print(f"  non-rank18          : {len(active_non18)}")
print(f"  rank18              : {len(active_r18)}")
print(f"HOLD total            : {len(held)}")
print(f"  rank18              : {len(held_r18)}")

total = len(processed) + len(active) + len(held)

print(f"\nTOTAL YAML ACCOUNTING : {total}/80")
print(
    "RANK18 ACCOUNTING    :",
    len(processed_r18) + len(active_r18) + len(held_r18),
    "/10"
)

# invariants
assert total == 80, f"ERROR: accounting is {total}, expected 80"
assert len(processed_non18) == 70, (
    f"ERROR: expected 70 completed non-rank18, found {len(processed_non18)}"
)
assert len(active_non18) == 0, (
    f"ERROR: non-rank18 config still active: {[p.name for p in active_non18]}"
)
assert len(held_non18) == 0
assert len(processed_r18) + len(active_r18) + len(held_r18) == 10

print("\nACTIVE RANK18 CONFIGS:")
for p in active_r18:
    print(" ", p.name)

print("\nPROCESSED RANK18 CONFIGS:")
for p in processed_r18:
    print(" ", p.name)

# ------------------------------------------------------------------
# 2. Check whether an old training process is still alive
# ------------------------------------------------------------------

ps_cmd = r"""
Get-CimInstance Win32_Process |
Where-Object {
    $_.CommandLine -match 'nh_run\.py' -and
    $_.CommandLine -match 'top18_additional8_selected_epoch_train_v57'
} |
Select-Object ProcessId, CreationDate, CommandLine |
Format-List
"""

res = subprocess.run(
    ["powershell", "-NoProfile", "-Command", ps_cmd],
    capture_output=True,
    text=True
)

print("\n" + "=" * 100)
print("CURRENT NEURALHYDROLOGY PROCESSES")
print("=" * 100)

if res.stdout.strip():
    print(res.stdout)
    print("WARNING: training process is still alive.")
else:
    print("None. Good.")

# ------------------------------------------------------------------
# 3. Inspect latest rank18 run directories / logs
# ------------------------------------------------------------------

rank18_dirs = sorted(
    [p for p in RUN_DIR.glob("*rank18*") if p.is_dir()],
    key=lambda p: p.stat().st_mtime,
    reverse=True
)

print("\n" + "=" * 100)
print("LATEST RANK18 RUNS")
print("=" * 100)

for rd in rank18_dirs[:6]:

    print("\n---", rd.name)
    print("Modified:", rd.stat().st_mtime)

    log = rd / "output.log"

    if log.exists():
        try:
            lines = log.read_text(
                encoding="utf-8",
                errors="replace"
            ).splitlines()

            for line in lines[-8:]:
                print(line)

        except Exception as e:
            print("Could not read log:", e)
    else:
        print("No output.log")

print("\n" + "=" * 100)
print("AUDIT PASSED")
print("=" * 100)

RANK18 POST-STOP AUDIT

FILESYSTEM:
Processed total       : 71
  non-rank18          : 70
  rank18              : 1
Active total          : 9
  non-rank18          : 0
  rank18              : 9
HOLD total            : 0
  rank18              : 0

TOTAL YAML ACCOUNTING : 80/80
RANK18 ACCOUNTING    : 10 /10

ACTIVE RANK18 CONFIGS:
  Top18_Add8_rank18_1708_155828_SE30_RS191889.yml
  Top18_Add8_rank18_1708_155828_SE30_RS307443.yml
  Top18_Add8_rank18_1708_155828_SE30_RS328237.yml
  Top18_Add8_rank18_1708_155828_SE30_RS385445.yml
  Top18_Add8_rank18_1708_155828_SE30_RS473461.yml
  Top18_Add8_rank18_1708_155828_SE30_RS643969.yml
  Top18_Add8_rank18_1708_155828_SE30_RS845563.yml
  Top18_Add8_rank18_1708_155828_SE30_RS985957.yml
  Top18_Add8_rank18_1708_155828_SE30_RS993061.yml

PROCESSED RANK18 CONFIGS:
  Top18_Add8_rank18_1708_155828_SE30_RS665980.yml

CURRENT NEURALHYDROLOGY PROCESSES
None. Good.

LATEST RANK18 RUNS

--- Top18_Add8_rank18_1708_155828_SE30_RS473461_0709_190550
Modified: 1788

In [1]:
import sys
import torch
from pathlib import Path

CONFIG_DIR = Path(
    r"F:\Experiments\CAMELS_US\Clean_4_upload"
    r"\Revision_2026_Top18_4Framework"
    r"\configs\top18_additional8_selected_epoch_train_v57"
)

PROCESSED_DIR = CONFIG_DIR / "processed"

print("Python         :", sys.executable)
print("PyTorch        :", torch.__version__)
print("CUDA available :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU 0          :", torch.cuda.get_device_name(0))

processed = sorted(PROCESSED_DIR.glob("*.yml"))
active = sorted(CONFIG_DIR.glob("*.yml"))

print("\nProcessed :", len(processed))
print("Active    :", len(active))
print("Total     :", len(processed) + len(active))

processed_r18 = [p for p in processed if "rank18_" in p.name]
active_r18 = [p for p in active if "rank18_" in p.name]

print("Processed rank18 :", len(processed_r18))
print("Active rank18    :", len(active_r18))

assert "WindowsApps" not in str(sys.executable)
assert torch.cuda.is_available()
assert len(processed) + len(active) == 80
assert len(active_r18) == len(active)

Python         : c:\Users\hosseinifh\AppData\Local\anaconda3\envs\MyPy_env2025\python.exe
PyTorch        : 2.7.1+cu118
CUDA available : True
GPU 0          : NVIDIA GeForce RTX 3060

Processed : 71
Active    : 9
Total     : 80
Processed rank18 : 1
Active rank18    : 9


In [ ]:
from neuralhydrology.nh_run_scheduler import schedule_runs

print("=" * 90)
print("STARTING REMAINING RANK18 RUNS")
print("Concurrency = 1 | GPU = 0")
print("=" * 90)

schedule_runs(
    mode="train",
    directory=CONFIG_DIR,
    gpu_ids=[0],
    runs_per_gpu=1,
)

STARTING REMAINING RANK18 RUNS
Concurrency = 1 | GPU = 0
Starting run 1/9: python C:\Users\hosseinifh\neuralhydrology\neuralhydrology\nh_run.py train --config-file F:\Experiments\CAMELS_US\Clean_4_upload\Revision_2026_Top18_4Framework\configs\top18_additional8_selected_epoch_train_v57\Top18_Add8_rank18_1708_155828_SE30_RS845563.yml --gpu 0
Finished run python C:\Users\hosseinifh\neuralhydrology\neuralhydrology\nh_run.py train --config-file F:\Experiments\CAMELS_US\Clean_4_upload\Revision_2026_Top18_4Framework\configs\top18_additional8_selected_epoch_train_v57\Top18_Add8_rank18_1708_155828_SE30_RS845563.yml --gpu 0
Cleaning up...


Moved F:\Experiments\CAMELS_US\Clean_4_upload\Revision_2026_Top18_4Framework\configs\top18_additional8_selected_epoch_train_v57\Top18_Add8_rank18_1708_155828_SE30_RS845563.yml into directory of processed configs at F:\Experiments\CAMELS_US\Clean_4_upload\Revision_2026_Top18_4Framework\configs\top18_additional8_selected_epoch_train_v57\processed\Top18_Add8_ran

In [1]:
from pathlib import Path
import sys
import torch

CONFIG_DIR = Path(
    r"F:\Experiments\CAMELS_US\Clean_4_upload"
    r"\Revision_2026_Top18_4Framework"
    r"\configs\top18_additional8_selected_epoch_train_v57"
)

PROCESSED_DIR = CONFIG_DIR / "processed"

print("=" * 90)
print("POST-RESET FINAL RANK18 AUDIT")
print("=" * 90)

print("Python         :", sys.executable)
print("PyTorch        :", torch.__version__)
print("CUDA available :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU 0          :", torch.cuda.get_device_name(0))

processed = sorted(PROCESSED_DIR.glob("*.yml"))
active = sorted(CONFIG_DIR.glob("*.yml"))

processed_r18 = [p for p in processed if "rank18_" in p.name]
active_r18 = [p for p in active if "rank18_" in p.name]

processed_non18 = [p for p in processed if "rank18_" not in p.name]
active_non18 = [p for p in active if "rank18_" not in p.name]

print("\nACCOUNTING")
print("Processed total     :", len(processed))
print("  non-rank18        :", len(processed_non18))
print("  rank18            :", len(processed_r18))
print("Active total        :", len(active))
print("  non-rank18        :", len(active_non18))
print("  rank18            :", len(active_r18))
print("TOTAL               :", len(processed) + len(active))

print("\nACTIVE CONFIGS:")
for p in active:
    print(" ", p.name)

assert "WindowsApps" not in str(sys.executable)
assert torch.cuda.is_available()
assert len(processed_non18) == 70
assert len(active_non18) == 0
assert len(processed) + len(active) == 80
assert len(processed_r18) + len(active_r18) == 10

print("\nAUDIT PASS")

POST-RESET FINAL RANK18 AUDIT
Python         : c:\Users\hosseinifh\AppData\Local\anaconda3\envs\MyPy_env2025\python.exe
PyTorch        : 2.7.1+cu118
CUDA available : True
GPU 0          : NVIDIA GeForce RTX 3060

ACCOUNTING
Processed total     : 78
  non-rank18        : 70
  rank18            : 8
Active total        : 2
  non-rank18        : 0
  rank18            : 2
TOTAL               : 80

ACTIVE CONFIGS:
  Top18_Add8_rank18_1708_155828_SE30_RS307443.yml
  Top18_Add8_rank18_1708_155828_SE30_RS385445.yml

AUDIT PASS


In [3]:
from neuralhydrology.nh_run_scheduler import schedule_runs

print("=" * 90)
print("FINAL TWO RANK18 RUNS")
print("Concurrency = 1 | GPU = 0")
print("=" * 90)

schedule_runs(
    mode="train",
    directory=CONFIG_DIR,
    gpu_ids=[0],
    runs_per_gpu=1,
)

FINAL TWO RANK18 RUNS
Concurrency = 1 | GPU = 0
Starting run 1/2: python C:\Users\hosseinifh\neuralhydrology\neuralhydrology\nh_run.py train --config-file F:\Experiments\CAMELS_US\Clean_4_upload\Revision_2026_Top18_4Framework\configs\top18_additional8_selected_epoch_train_v57\Top18_Add8_rank18_1708_155828_SE30_RS307443.yml --gpu 0
Finished run python C:\Users\hosseinifh\neuralhydrology\neuralhydrology\nh_run.py train --config-file F:\Experiments\CAMELS_US\Clean_4_upload\Revision_2026_Top18_4Framework\configs\top18_additional8_selected_epoch_train_v57\Top18_Add8_rank18_1708_155828_SE30_RS307443.yml --gpu 0
Cleaning up...


Moved F:\Experiments\CAMELS_US\Clean_4_upload\Revision_2026_Top18_4Framework\configs\top18_additional8_selected_epoch_train_v57\Top18_Add8_rank18_1708_155828_SE30_RS307443.yml into directory of processed configs at F:\Experiments\CAMELS_US\Clean_4_upload\Revision_2026_Top18_4Framework\configs\top18_additional8_selected_epoch_train_v57\processed\Top18_Add8_rank18_1708_

In [5]:
from pathlib import Path
import re

ROOT = Path(
    r"F:\Experiments\CAMELS_US\Clean_4_upload"
    r"\Revision_2026_Top18_4Framework"
)

CONFIG_DIR = ROOT / "configs" / "top18_additional8_selected_epoch_train_v57"
PROCESSED_DIR = CONFIG_DIR / "processed"
RUN_DIR = ROOT / "runs" / "top18_additional8_selected_epoch_train_v57"

print("=" * 110)
print("FINAL 81-RUN DUPLICATE AUDIT")
print("=" * 110)

# ------------------------------------------------------------------
# 1. Canonical expected logical runs = 80 processed YAMLs
# ------------------------------------------------------------------

processed_cfgs = sorted(PROCESSED_DIR.glob("*.yml"))
expected_ids = {p.stem for p in processed_cfgs}

print("Processed YAMLs :", len(processed_cfgs))
assert len(processed_cfgs) == 80, (
    f"Expected 80 processed configs, found {len(processed_cfgs)}"
)

# ------------------------------------------------------------------
# 2. Parse run directory -> logical config ID
# Run dirs end with e.g. _0209_175418
# ------------------------------------------------------------------

run_dirs = sorted([
    p for p in RUN_DIR.iterdir()
    if p.is_dir() and (p / "config.yml").is_file()
])

timestamp_re = re.compile(r"_(\d{4})_(\d{6})$")

def logical_id_from_run_dir(run_dir):
    return timestamp_re.sub("", run_dir.name)

records = []

for rd in run_dirs:
    logical_id = logical_id_from_run_dir(rd)

    # expected selected epoch from SE20 / SE25 / SE30 in config name
    m_se = re.search(r"_SE(\d+)_", logical_id)
    expected_epoch = int(m_se.group(1)) if m_se else None

    # inspect output.log
    log_file = rd / "output.log"
    max_logged_epoch = None
    tail = []

    if log_file.exists():
        text = log_file.read_text(encoding="utf-8", errors="replace")

        epochs = [
            int(x)
            for x in re.findall(r"Epoch\s+(\d+)\s+average loss", text)
        ]

        if epochs:
            max_logged_epoch = max(epochs)

        tail = text.splitlines()[-5:]

    # inspect checkpoints
    checkpoints = sorted(rd.glob("model_epoch*.pt"))

    checkpoint_epochs = []
    for cp in checkpoints:
        m = re.search(r"model_epoch(\d+)\.pt$", cp.name)
        if m:
            checkpoint_epochs.append(int(m.group(1)))

    max_checkpoint_epoch = (
        max(checkpoint_epochs) if checkpoint_epochs else None
    )

    final_checkpoint = (
        rd / f"model_epoch{expected_epoch:03d}.pt"
        if expected_epoch is not None else None
    )

    final_checkpoint_exists = (
        final_checkpoint.is_file()
        if final_checkpoint is not None else False
    )

    records.append({
        "run_dir": rd,
        "logical_id": logical_id,
        "expected_epoch": expected_epoch,
        "max_logged_epoch": max_logged_epoch,
        "max_checkpoint_epoch": max_checkpoint_epoch,
        "final_checkpoint_exists": final_checkpoint_exists,
        "mtime": rd.stat().st_mtime,
        "tail": tail,
    })

print("Valid run dirs  :", len(records))

# ------------------------------------------------------------------
# 3. Group by logical model/config
# ------------------------------------------------------------------

groups = {}

for r in records:
    groups.setdefault(r["logical_id"], []).append(r)

duplicates = {
    k: v
    for k, v in groups.items()
    if len(v) > 1
}

missing = sorted(expected_ids - set(groups))
unexpected = sorted(set(groups) - expected_ids)

print("Unique logical IDs :", len(groups))
print("Duplicate IDs      :", len(duplicates))
print("Missing IDs        :", len(missing))
print("Unexpected IDs     :", len(unexpected))

print("\n" + "=" * 110)
print("DUPLICATE DETAILS")
print("=" * 110)

if not duplicates:
    print("NO DUPLICATE LOGICAL CONFIG FOUND.")
else:
    for logical_id, items in duplicates.items():

        print(f"\nLOGICAL CONFIG: {logical_id}")
        print(f"Copies        : {len(items)}")

        for i, r in enumerate(
            sorted(items, key=lambda x: x["mtime"]),
            start=1
        ):
            complete_by_log = (
                r["expected_epoch"] is not None
                and r["max_logged_epoch"] is not None
                and r["max_logged_epoch"] >= r["expected_epoch"]
            )

            print("\n  COPY", i)
            print("  Folder                 :", r["run_dir"].name)
            print("  Expected epoch         :", r["expected_epoch"])
            print("  Max logged epoch       :", r["max_logged_epoch"])
            print("  Max checkpoint epoch   :", r["max_checkpoint_epoch"])
            print("  Final checkpoint exists:", r["final_checkpoint_exists"])
            print("  Complete by log        :", complete_by_log)

            print("  Log tail:")
            for line in r["tail"]:
                print("     ", line)

print("\n" + "=" * 110)
print("GLOBAL CONSISTENCY")
print("=" * 110)

if missing:
    print("\nMISSING:")
    for x in missing:
        print(" ", x)
else:
    print("Missing    : NONE")

if unexpected:
    print("\nUNEXPECTED:")
    for x in unexpected:
        print(" ", x)
else:
    print("Unexpected : NONE")

print("\nExpected configs :", len(expected_ids))
print("Logical run IDs  :", len(groups))
print("Physical folders :", len(records))

assert len(expected_ids) == 80
assert len(groups) == 80, (
    f"Expected 80 unique logical runs, found {len(groups)}"
)
assert len(records) == 81
assert len(duplicates) == 1, (
    f"Expected exactly one duplicated logical run, found {len(duplicates)}"
)

print("\nAUDIT COMPLETE — EXACTLY ONE LOGICAL RUN IS DUPLICATED.")

FINAL 81-RUN DUPLICATE AUDIT
Processed YAMLs : 80
Valid run dirs  : 81
Unique logical IDs : 80
Duplicate IDs      : 1
Missing IDs        : 0
Unexpected IDs     : 0

DUPLICATE DETAILS

LOGICAL CONFIG: Top18_Add8_rank18_1708_155828_SE30_RS307443
Copies        : 2

  COPY 1
  Folder                 : Top18_Add8_rank18_1708_155828_SE30_RS307443_1009_041139
  Expected epoch         : 30
  Max logged epoch       : 26
  Max checkpoint epoch   : None
  Final checkpoint exists: False
  Complete by log        : False
  Log tail:
      2026-09-10 09:44:39,350: Epoch 23 average loss: avg_loss: 0.16918, avg_total_loss: 0.16918
      2026-09-10 09:59:07,976: Epoch 24 average loss: avg_loss: 0.16745, avg_total_loss: 0.16745
      2026-09-10 09:59:07,976: Setting learning rate to 0.0001
      2026-09-10 10:13:36,097: Epoch 25 average loss: avg_loss: 0.15083, avg_total_loss: 0.15083
      2026-09-10 10:28:04,837: Epoch 26 average loss: avg_loss: 0.14492, avg_total_loss: 0.14492

  COPY 2
  Folder      

In [6]:
from pathlib import Path
import shutil

ROOT = Path(
    r"F:\Experiments\CAMELS_US\Clean_4_upload"
    r"\Revision_2026_Top18_4Framework"
)

RUN_DIR = ROOT / "runs" / "top18_additional8_selected_epoch_train_v57"

BAD_RUN = RUN_DIR / "Top18_Add8_rank18_1708_155828_SE30_RS307443_1009_041139"

QUARANTINE_DIR = ROOT / "quarantine" / "incomplete_training_runs"
QUARANTINE_DIR.mkdir(parents=True, exist_ok=True)

DEST = QUARANTINE_DIR / BAD_RUN.name

assert BAD_RUN.is_dir(), f"Bad run not found: {BAD_RUN}"
assert not DEST.exists(), f"Quarantine destination already exists: {DEST}"

shutil.move(str(BAD_RUN), str(DEST))

print("Moved incomplete run to quarantine:")
print(" ", DEST)

Moved incomplete run to quarantine:
  F:\Experiments\CAMELS_US\Clean_4_upload\Revision_2026_Top18_4Framework\quarantine\incomplete_training_runs\Top18_Add8_rank18_1708_155828_SE30_RS307443_1009_041139


In [7]:
from neuralhydrology.utils.config import Config

run_dirs = sorted([
    p for p in RUN_DIR.iterdir()
    if p.is_dir() and (p / "config.yml").is_file()
])

print("Physical valid runs:", len(run_dirs))

errors = []

for rd in run_dirs:
    cfg = Config(rd / "config.yml")
    epoch = int(cfg.epochs)

    model_cp = rd / f"model_epoch{epoch:03d}.pt"
    optimizer_cp = rd / f"optimizer_state_epoch{epoch:03d}.pt"

    if not model_cp.is_file():
        errors.append((rd.name, "missing model", epoch))

    if not optimizer_cp.is_file():
        errors.append((rd.name, "missing optimizer", epoch))

print("Checkpoint errors:", len(errors))

for x in errors[:20]:
    print(x)

assert len(run_dirs) == 80
assert not errors

print("\n========================================")
print("FINAL TRAINING AUDIT: PASS")
print("80/80 unique runs")
print("80/80 final selected-epoch checkpoints")
print("========================================")

Physical valid runs: 80
Checkpoint errors: 0

FINAL TRAINING AUDIT: PASS
80/80 unique runs
80/80 final selected-epoch checkpoints


In [9]:
from pathlib import Path

EVAL_DATA_DIR = Path(r"D:\Datesets\CAMELS_US")

print("CAMELS root:", EVAL_DATA_DIR)
print("Exists     :", EVAL_DATA_DIR.exists())

required = [
    EVAL_DATA_DIR / "camels_attributes_v2.0",
    EVAL_DATA_DIR / "basin_mean_forcing" / "daymet",
    EVAL_DATA_DIR / "basin_mean_forcing" / "maurer",
    EVAL_DATA_DIR / "basin_mean_forcing" / "nldas",
    EVAL_DATA_DIR / "usgs_streamflow",
]

for p in required:
    print(f"{p.exists():5}  {p}")

assert EVAL_DATA_DIR.is_dir(), EVAL_DATA_DIR
assert all(p.is_dir() for p in required), "CAMELS-US root is incomplete."

print("\nCAMELS-US DATA ROOT: PASS")

CAMELS root: D:\Datesets\CAMELS_US
Exists     : True
    1  D:\Datesets\CAMELS_US\camels_attributes_v2.0
    1  D:\Datesets\CAMELS_US\basin_mean_forcing\daymet
    1  D:\Datesets\CAMELS_US\basin_mean_forcing\maurer
    1  D:\Datesets\CAMELS_US\basin_mean_forcing\nldas
    1  D:\Datesets\CAMELS_US\usgs_streamflow

CAMELS-US DATA ROOT: PASS


In [10]:
from pathlib import Path
import yaml
import shutil
import subprocess
import sys
import neuralhydrology

ROOT = Path(
    r"F:\Experiments\CAMELS_US\Clean_4_upload"
    r"\Revision_2026_Top18_4Framework"
)

RUN_DIR = ROOT / "runs" / "top18_additional8_selected_epoch_train_v57"

BASIN_FILE = Path(
    r"F:\Experiments\CAMELS_US\Clean_4_upload\531_basin_list.txt"
)

run_dirs = sorted([
    p for p in RUN_DIR.iterdir()
    if p.is_dir() and (p / "config.yml").is_file()
])

assert len(run_dirs) == 80

# Take first valid run only
rd = run_dirs[0]

cfg_path = rd / "config.yml"
backup = rd / "config_before_test.yml"

if not backup.exists():
    shutil.copy2(cfg_path, backup)

cfg = yaml.safe_load(cfg_path.read_text(encoding="utf-8"))

cfg["data_dir"] = str(EVAL_DATA_DIR)
cfg["test_basin_file"] = str(BASIN_FILE)

with cfg_path.open("w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

epoch = int(cfg["epochs"])

NH_RUN = (
    Path(neuralhydrology.__file__).resolve().parent
    / "nh_run.py"
)

cmd = [
    sys.executable,
    str(NH_RUN),
    "evaluate",
    "--run-dir", str(rd),
    "--period", "test",
    "--epoch", str(epoch),
    "--gpu", "0",
]

print("SMOKE TEST")
print("Run   :", rd.name)
print("Epoch :", epoch)
print("Data  :", EVAL_DATA_DIR)
print()
print(" ".join(cmd))

result = subprocess.run(cmd)

print("\nExit code:", result.returncode)

assert result.returncode == 0, "Smoke test failed."

print("\nSMOKE TEST: PASS")

SMOKE TEST
Run   : Top18_Add8_rank10_1206_203556_SE20_RS191889_0609_203502
Epoch : 20
Data  : D:\Datesets\CAMELS_US

c:\Users\hosseinifh\AppData\Local\anaconda3\envs\MyPy_env2025\python.exe C:\Users\hosseinifh\neuralhydrology\neuralhydrology\nh_run.py evaluate --run-dir F:\Experiments\CAMELS_US\Clean_4_upload\Revision_2026_Top18_4Framework\runs\top18_additional8_selected_epoch_train_v57\Top18_Add8_rank10_1206_203556_SE20_RS191889_0609_203502 --period test --epoch 20 --gpu 0

Exit code: 0

SMOKE TEST: PASS


In [11]:
from pathlib import Path
import pandas as pd

PICKLES_DIR = Path(
    r"F:\Experiments\CAMELS_US\Clean_4_upload\Pickles"
)

assert PICKLES_DIR.is_dir(), PICKLES_DIR

files = sorted([
    p for p in PICKLES_DIR.rglob("*")
    if p.is_file()
])

rows = []

for p in files:
    rows.append({
        "relative_path": str(p.relative_to(PICKLES_DIR)),
        "suffix": p.suffix.lower(),
        "size_MB": round(p.stat().st_size / 1024**2, 3),
        "modified": pd.Timestamp(p.stat().st_mtime, unit="s"),
    })

inventory = pd.DataFrame(rows)

print("=" * 110)
print("PICKLES INVENTORY")
print("=" * 110)

print("Root :", PICKLES_DIR)
print("Files:", len(inventory))

display(inventory)

print("\nLIKELY RELEVANT FILES")
mask = inventory["relative_path"].str.contains(
    r"obs|test|top.?10|ensemble|metric|nse|kge|prediction|pred",
    case=False,
    regex=True,
    na=False,
)

display(inventory.loc[mask].reset_index(drop=True))

PICKLES INVENTORY
Root : F:\Experiments\CAMELS_US\Clean_4_upload\Pickles
Files: 27


,relative_path,suffix,size_MB,modified
0,All_Cluster_wise_Configs_CAMELSUS_test.p,.p,6161.913,2025-04-22 09:03:11.957734346
1,All_Top10_Configs_CAMELSUS_test.p,.p,3734.512,2025-04-22 08:59:15.816460848
2,ensemble_Cluster_wise_CAMELSUS_test.p,.p,373.723,2026-02-04 10:22:18.778712988
3,ensemble_Top10_CAMELSUS_test.p,.p,373.723,2026-02-04 10:21:09.467375994
4,global_event_catalog.csv,.csv,2.199,2026-02-05 21:42:20.773970127
5,global_event_index.p,.p,2.549,2026-02-05 21:42:20.884551048
6,highflow_events_log.csv,.csv,2.148,2026-02-05 21:42:18.101852655
7,highflow_events_log_with_global.csv,.csv,2.199,2026-02-05 21:42:20.634393454
8,highflow_events_run.log,.log,0.410,2026-02-05 21:42:30.100815296
9,metrics_all_test_period.p,.p,3.709,2026-02-04 12:58:35.450358868



LIKELY RELEVANT FILES


,relative_path,suffix,size_MB,modified
0,All_Cluster_wise_Configs_CAMELSUS_test.p,.p,6161.913,2025-04-22 09:03:11.957734346
1,All_Top10_Configs_CAMELSUS_test.p,.p,3734.512,2025-04-22 08:59:15.816460848
2,ensemble_Cluster_wise_CAMELSUS_test.p,.p,373.723,2026-02-04 10:22:18.778712988
3,ensemble_Top10_CAMELSUS_test.p,.p,373.723,2026-02-04 10:21:09.467375994
4,metrics_all_test_period.p,.p,3.709,2026-02-04 12:58:35.450358868
5,metrics_by_water_year.p,.p,37.060,2026-02-04 16:13:41.095117092
6,metrics_by_year_season.p,.p,146.921,2026-02-05 04:02:01.570993900
7,metrics_highflow_events_allbasins.p,.p,2.636,2026-02-05 21:42:20.932072401
8,metrics_highflow_events_eventwise.p,.p,117.657,2026-02-05 21:42:30.084494352
9,metrics_highflow_events_normalized_bins.p,.p,1172.567,2026-02-06 21:31:12.753504992


In [16]:
from pathlib import Path
import sys
import yaml
import shutil
import subprocess
import time
import neuralhydrology

# ============================================================
# PATHS
# ============================================================

EVAL_DATA_DIR = Path(r"D:\Datesets\CAMELS_US")

BASIN_FILE = Path(
    r"F:\Experiments\CAMELS_US\Clean_4_upload\531_basin_list.txt"
)

ROOT = Path(
    r"F:\Experiments\CAMELS_US\Clean_4_upload"
    r"\Revision_2026_Top18_4Framework"
)

RUN_DIR = (
    ROOT / "runs" / "top18_additional8_selected_epoch_train_v57"
)

NH_RUN = (
    Path(neuralhydrology.__file__).resolve().parent
    / "nh_run.py"
)

MAX_PARALLEL = 2
GPU = 0

# ============================================================
# 1. AUDIT 80 RUNS
# ============================================================

run_dirs = sorted([
    p for p in RUN_DIR.iterdir()
    if p.is_dir() and (p / "config.yml").is_file()
])

assert len(run_dirs) == 80, len(run_dirs)

jobs = []
already_done = []

for rd in run_dirs:

    cfg_path = rd / "config.yml"

    # Preserve original training config once
    backup = rd / "config_before_test.yml"
    if not backup.exists():
        shutil.copy2(cfg_path, backup)

    cfg = yaml.safe_load(
        cfg_path.read_text(encoding="utf-8")
    )

    epoch = int(cfg["epochs"])

    # Point evaluation to canonical raw CAMELS-US
    cfg["data_dir"] = str(EVAL_DATA_DIR)
    cfg["test_basin_file"] = str(BASIN_FILE)

    with cfg_path.open("w", encoding="utf-8") as f:
        yaml.safe_dump(
            cfg,
            f,
            sort_keys=False,
            allow_unicode=True
        )

    result_file = (
        rd / "test"
        / f"model_epoch{epoch:03d}"
        / "test_results.p"
    )

    if result_file.is_file():
        already_done.append(
            (rd.name, epoch, result_file)
        )
    else:
        jobs.append(
            (rd, epoch, result_file)
        )

print("=" * 100)
print("TOP18 ADD8 DIRECT TEST")
print("=" * 100)

print("Valid trained runs :", len(run_dirs))
print("Already tested     :", len(already_done))
print("Remaining tests    :", len(jobs))
print("Parallel tests     :", MAX_PARALLEL)

assert len(already_done) + len(jobs) == 80

# ============================================================
# 2. TEST QUEUE
# ============================================================

queue = list(jobs)
running = {}
failures = []
completed_now = 0

while queue or running:

    # launch
    while queue and len(running) < MAX_PARALLEL:

        rd, epoch, result_file = queue.pop(0)

        log_file = (
            rd / f"direct_test_epoch{epoch:03d}.log"
        )

        fh = log_file.open(
            "w",
            encoding="utf-8"
        )

        cmd = [
            sys.executable,
            str(NH_RUN),
            "evaluate",
            "--run-dir", str(rd),
            "--period", "test",
            "--epoch", str(epoch),
            "--gpu", str(GPU),
        ]

        print(
            f"START  "
            f"{80 - len(queue) - len(running):02d}/80  "
            f"{rd.name}  epoch={epoch}"
        )

        proc = subprocess.Popen(
            cmd,
            stdout=fh,
            stderr=subprocess.STDOUT,
        )

        running[proc] = {
            "run": rd,
            "epoch": epoch,
            "result": result_file,
            "log": log_file,
            "fh": fh,
        }

    # poll
    finished = []

    for proc, meta in list(running.items()):

        code = proc.poll()

        if code is None:
            continue

        meta["fh"].close()

        rd = meta["run"]
        epoch = meta["epoch"]
        result_file = meta["result"]

        if code == 0 and result_file.is_file():

            completed_now += 1

            print(
                f"DONE   {rd.name}  "
                f"epoch={epoch}"
            )

        else:

            failures.append({
                "run": rd.name,
                "epoch": epoch,
                "exit_code": code,
                "result_exists": result_file.is_file(),
                "log": str(meta["log"]),
            })

            print(
                f"FAILED {rd.name}  "
                f"epoch={epoch} exit={code}"
            )

        finished.append(proc)

    for proc in finished:
        running.pop(proc)

    if running:
        time.sleep(2)

# ============================================================
# 3. FINAL AUDIT
# ============================================================

tested = []

for rd in run_dirs:

    cfg = yaml.safe_load(
        (rd / "config.yml").read_text(encoding="utf-8")
    )

    epoch = int(cfg["epochs"])

    p = (
        rd / "test"
        / f"model_epoch{epoch:03d}"
        / "test_results.p"
    )

    if p.is_file():
        tested.append(p)

print("\n" + "=" * 100)
print("FINAL TEST AUDIT")
print("=" * 100)

print("Test results present :", len(tested))
print("Failures             :", len(failures))

if failures:
    for x in failures:
        print(x)

assert len(tested) == 80, (
    f"Expected 80 test_results.p, found {len(tested)}"
)

assert not failures

print("\nDIRECT TEST: 80/80 PASS")

TOP18 ADD8 DIRECT TEST
Valid trained runs : 80
Already tested     : 1
Remaining tests    : 79
Parallel tests     : 2
START  02/80  Top18_Add8_rank10_1206_203556_SE20_RS307443_0309_123220  epoch=20
START  02/80  Top18_Add8_rank10_1206_203556_SE20_RS328237_0109_010409  epoch=20
DONE   Top18_Add8_rank10_1206_203556_SE20_RS307443_0309_123220  epoch=20
START  03/80  Top18_Add8_rank10_1206_203556_SE20_RS385445_2208_062449  epoch=20
DONE   Top18_Add8_rank10_1206_203556_SE20_RS328237_0109_010409  epoch=20
START  04/80  Top18_Add8_rank10_1206_203556_SE20_RS473461_0409_065249  epoch=20
DONE   Top18_Add8_rank10_1206_203556_SE20_RS385445_2208_062449  epoch=20
START  05/80  Top18_Add8_rank10_1206_203556_SE20_RS643969_0509_220920  epoch=20
DONE   Top18_Add8_rank10_1206_203556_SE20_RS473461_0409_065249  epoch=20
START  06/80  Top18_Add8_rank10_1206_203556_SE20_RS665980_2008_030021  epoch=20
DONE   Top18_Add8_rank10_1206_203556_SE20_RS643969_0509_220920  epoch=20
START  07/80  Top18_Add8_rank10_1206_2

# Stage B — one validation after training, then test

**Do not run this stage yet.**

This stage intentionally does not block training.

When all 80 runs are finished, set the real raw CAMELS-US evaluation root in `EVAL_DATA_DIR_OVERRIDE`. It must contain:

- `camels_attributes_v2.0`
- `basin_mean_forcing/daymet`
- `basin_mean_forcing/maurer`
- `basin_mean_forcing/nldas`
- `usgs_streamflow`

Then set `RUN_POSTTRAIN_EVALUATION=True`.

The notebook will:
1. point the saved run configs to the raw evaluation data;
2. run validation once at each run's selected epoch;
3. only if all validation jobs succeed, run test once at the same selected epoch.


In [ ]:
def is_complete_eval_camels_root(p: Path) -> bool:
    required = [
        p / "camels_attributes_v2.0",
        p / "basin_mean_forcing" / "daymet",
        p / "basin_mean_forcing" / "maurer",
        p / "basin_mean_forcing" / "nldas",
        p / "usgs_streamflow",
    ]
    return p.is_dir() and all(x.is_dir() for x in required)


def run_evaluation_jobs(
    run_dirs: list[Path],
    period: str,
    max_parallel: int = 3,
    gpu_id: int = 0,
):
    """
    Launch NeuralHydrology's own nh_run.py evaluate command,
    but explicitly pass validation/test period and the run-specific selected epoch.
    """
    assert period in {"validation", "test"}

    nh_run_script = (
        Path(neuralhydrology.__file__).resolve().parent / "nh_run.py"
    )

    queue = list(run_dirs)
    running = {}
    failures = []

    while queue or running:
        while queue and len(running) < max_parallel:
            rd = queue.pop(0)
            cfg = Config(rd / "config.yml")
            epoch = int(cfg.epochs)

            log_path = rd / f"posttrain_{period}_epoch{epoch:03d}.log"
            log_handle = log_path.open("w", encoding="utf-8")

            cmd = [
                sys.executable,
                str(nh_run_script),
                "evaluate",
                "--run-dir", str(rd),
                "--period", period,
                "--epoch", str(epoch),
                "--gpu", str(gpu_id),
            ]

            print(
                f"Starting {period}: {rd.name} | epoch={epoch}"
            )
            proc = subprocess.Popen(
                cmd,
                stdout=log_handle,
                stderr=subprocess.STDOUT,
            )
            running[proc] = (rd, epoch, log_handle)

        finished = []
        for proc, meta in list(running.items()):
            code = proc.poll()
            if code is not None:
                rd, epoch, log_handle = meta
                log_handle.close()

                if code != 0:
                    failures.append(
                        (str(rd), epoch, int(code))
                    )
                    print(
                        f"FAILED {period}: {rd.name} | "
                        f"epoch={epoch} | exit={code}"
                    )
                else:
                    print(
                        f"Finished {period}: {rd.name} | epoch={epoch}"
                    )
                finished.append(proc)

        for proc in finished:
            running.pop(proc, None)

        if running:
            time.sleep(2)

    if failures:
        raise RuntimeError(
            f"{len(failures)} {period} evaluation job(s) failed. "
            f"Inspect posttrain_{period}_epochXXX.log in the corresponding run dirs."
        )


if RUN_POSTTRAIN_EVALUATION:
    run_dirs = find_run_dirs(RUN_DIR)
    assert len(run_dirs) == 80, (
        f"Need 80 trained run directories before Stage B; found {len(run_dirs)}"
    )

    if EVAL_DATA_DIR_OVERRIDE is None:
        raise ValueError(
            "Set EVAL_DATA_DIR_OVERRIDE to the real raw CAMELS-US root "
            "before running Stage B."
        )

    EVAL_DATA_DIR = Path(EVAL_DATA_DIR_OVERRIDE).resolve()

    if not is_complete_eval_camels_root(EVAL_DATA_DIR):
        raise FileNotFoundError(
            f"Not a complete CAMELS-US evaluation root: {EVAL_DATA_DIR}"
        )

    # Patch only paths in saved run configs, preserving a training-config backup.
    for rd in run_dirs:
        config_path = rd / "config.yml"
        backup_path = rd / "config_stageA_training.yml"

        if not backup_path.exists():
            shutil.copy2(config_path, backup_path)

        cfg = yaml.safe_load(config_path.read_text(encoding="utf-8"))
        cfg["data_dir"] = str(EVAL_DATA_DIR)
        cfg["validation_basin_file"] = str(BASIN_FILE)
        cfg["test_basin_file"] = str(BASIN_FILE)

        with config_path.open("w", encoding="utf-8") as f:
            yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)

    print("Evaluation paths patched: PASS")

    print("\n" + "=" * 100)
    print("POST-TRAIN VALIDATION — ONCE PER RUN")
    print("=" * 100)
    run_evaluation_jobs(
        run_dirs=run_dirs,
        period="validation",
        max_parallel=3,
        gpu_id=0,
    )
    print("VALIDATION STAGE: PASS")

    print("\n" + "=" * 100)
    print("TEST — SAME SELECTED-EPOCH CHECKPOINT")
    print("=" * 100)
    run_evaluation_jobs(
        run_dirs=run_dirs,
        period="test",
        max_parallel=3,
        gpu_id=0,
    )
    print("TEST STAGE: PASS")
else:
    print("RUN_POSTTRAIN_EVALUATION=False")
    print("Correct for now: Stage B does not block training.")


## Final protocol summary

For each run:

**Stage A**
`cached training data → epochs 1…selected_epoch → save selected_epoch only`

No validation is performed while fitting.

After all 80 runs:

**Stage B**
`validation(selected_epoch) once → test(selected_epoch) once`

No new epoch selection is made at any point.
